In [1]:
import pandas as pd 
import numpy as np
from datetime import datetime, timedelta
import random 

In [ ]:
!pip show folktables
!pip show whyshift

# Pre-process Data

You can download the original data set from https://github.com/IrinaStatsLab/Awesome-CGM/wiki/Brown-(2019). 

In [ ]:
data = pd.read_csv("o_malley2021.csv")

In [11]:
def round_time_to_5min(dt):
    """Round datetime to nearest 5-minute interval"""
    # Include seconds in the rounding calculation
    total_seconds = dt.hour * 3600 + dt.minute * 60 + dt.second
    # Round to nearest 5-minute interval (300 seconds)
    rounded_seconds = round(total_seconds / 300) * 300
    # Calculate the rounded time
    hours = rounded_seconds // 3600
    minutes = (rounded_seconds % 3600) // 60
    # Handle case where rounding goes to next day
    if hours >= 24:
        # Move to next day and set to 00:00:00
        next_day = dt.replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=1)
        return next_day
    else:
        return dt.replace(hour=hours, minute=minutes, second=0, microsecond=0)

In [3]:
def create_complete_time_grid(start_time, end_time):
    """Create complete 5-minute time grid between start and end times"""
    time_grid = []
    current_time = start_time
    while current_time <= end_time:
        time_grid.append(current_time)
        current_time += timedelta(minutes=5)
    return time_grid

In [ ]:
data['time'] = pd.to_datetime(data['time'], format="mixed")
data['time'] = data['time'].apply(round_time_to_5min)
data_clean = data.drop_duplicates(subset=['id', 'time'], keep='first')
data_clean = data_clean.sort_values(['id', 'time']).reset_index(drop=True)
data_clean = data_clean[['id', 'time', 'gl', 'age', 'sex']]

In [ ]:
data_clean.to_csv("data_clean.csv", index=False)

In [62]:
def extract_glucose_traces(df, num_traces=10000, trace_duration_days=14):
    
    traces = []
    
    # Get patient metadata for quick lookup
    patient_info = df.groupby('id')[['age', 'sex']].first().to_dict('index')
    
    random.seed(0)
    for i in range(num_traces):
        
        # Randomly select a starting point from the dataset
        random_idx = random.randint(0, len(df) - 1)
        start_row = df.iloc[random_idx]
        
        selected_patient = start_row['id']
        selected_time = start_row['time']
        selected_date = selected_time.normalize()  # sets time to 00:00:00

        # Define two weeks
        before_start = selected_date - timedelta(days=7)
        before_end = selected_date - timedelta(minutes=5)  
        after_start = selected_date + timedelta(days=1)  
        after_end = selected_date + timedelta(days=7, hours=23, minutes=55) 
        
        # Get all data in the 2-week window
        
        window_data = df[
            (
                ((df['time'] >= before_start) & (df['time'] <= before_end)) |
                ((df['time'] >= after_start) & (df['time'] <= after_end))
            ) &
            (df['id'] == selected_patient)
        ].copy()
        
        # Check if the trace contains only one patient
        unique_patients = window_data['id'].unique()
        if len(unique_patients) == 1:
            
            # Create complete time grid
            before_time_grid = create_complete_time_grid(before_start, before_end)
            after_time_grid = create_complete_time_grid(after_start, after_end)
            complete_time_grid = before_time_grid + after_time_grid
            
            # Create DataFrame with complete time grid
            complete_trace = pd.DataFrame({
                'time': complete_time_grid,
                'id': selected_patient,
                'age': patient_info[selected_patient]['age'],
                'sex': patient_info[selected_patient]['sex']
            })
            
            # Merge with actual data to fill in glucose levels
            complete_trace = complete_trace.merge(
                window_data[['time', 'gl']], 
                left_on='time', 
                right_on='time', 
                how='left'
            )
            
            # Clean up columns
            complete_trace = complete_trace[['id', 'time', 'gl', 'age', 'sex']]
            
            # Add trace identifier
            complete_trace['trace_id'] = len(traces) + 1
            
            traces.append(complete_trace['gl'].to_numpy())
    
    return np.array(traces)


In [66]:
traces = extract_glucose_traces(data_clean, num_traces=10000)

In [103]:
na_counts = np.isnan(traces).sum(axis=1)
top_5000_indices = np.argsort(na_counts)[:5000]
selected_traces = traces[top_5000_indices]
#np.save('traces.npy', selected_traces)

In [104]:
def compress_traces(gl_values):
    assert len(gl_values) == 4032, "Trace must be of length 4032"
    
    # Reshape first and second halves into (7, 288)
    first_half = gl_values[:2016].reshape(7, 288)
    second_half = gl_values[2016:].reshape(7, 288)

    # Take mean across rows (i.e., column-wise mean) ⇒ (288,)
    first_compressed = np.nanmean(first_half, axis=0)
    second_compressed = np.nanmean(second_half, axis=0)

    return np.concatenate([first_compressed, second_compressed]) 

In [105]:
cgm_avg = np.array([compress_traces(row) for row in selected_traces])

In [106]:
cgm_avg = pd.DataFrame(cgm_avg)

In [107]:
cgm_avg

,0,1,2,3,4,5,6,7,8,9,...,566,567,568,569,570,571,572,573,574,575
0,218.142857,219.285714,220.571429,222.285714,224.000000,225.857143,227.428571,228.571429,229.857143,230.428571,...,226.000000,225.857143,225.857143,226.428571,227.142857,228.428571,230.000000,231.714286,233.571429,234.571429
1,102.142857,101.285714,101.714286,102.714286,103.857143,105.285714,105.571429,106.142857,107.428571,109.142857,...,157.285714,160.857143,165.000000,169.000000,171.857143,173.000000,172.571429,170.571429,167.714286,163.857143
2,174.000000,172.000000,169.714286,166.857143,164.142857,162.571429,162.000000,162.428571,163.857143,166.142857,...,168.571429,167.285714,167.285714,168.000000,169.142857,171.571429,175.285714,180.571429,186.142857,191.285714
3,152.142857,151.142857,150.285714,149.714286,149.714286,149.571429,149.000000,148.000000,147.000000,146.571429,...,167.000000,167.142857,167.000000,165.714286,163.857143,162.000000,159.714286,158.142857,157.285714,156.857143
4,119.714286,121.000000,121.714286,121.857143,121.571429,120.857143,120.857143,120.142857,119.571429,119.000000,...,146.285714,145.000000,143.428571,142.428571,141.714286,140.428571,139.571429,138.285714,137.428571,137.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,229.714286,224.428571,210.714286,204.428571,198.714286,190.800000,189.142857,186.428571,182.714286,181.142857,...,245.714286,245.142857,242.714286,241.000000,239.571429,238.142857,238.285714,238.000000,235.857143,233.285714
4996,149.714286,151.714286,153.857143,154.142857,152.857143,151.428571,151.571429,151.428571,152.857143,154.428571,...,181.166667,179.666667,181.333333,181.000000,179.166667,177.000000,175.000000,171.500000,168.833333,164.666667
4997,172.285714,173.000000,173.714286,174.285714,173.857143,173.142857,173.571429,173.285714,173.000000,172.571429,...,174.000000,175.000000,177.666667,178.333333,176.333333,175.000000,173.000000,169.500000,167.500000,164.333333
4998,203.428571,203.142857,200.714286,198.571429,197.571429,198.428571,200.428571,202.857143,204.571429,204.142857,...,189.285714,184.285714,180.285714,178.000000,175.714286,171.857143,166.714286,161.428571,157.000000,153.000000


In [108]:
cgm_avg.to_csv("cgm_avg.csv", index=False)